# 02 – Preview Processed Data

This notebook provides a quick sanity check of the cleaned Kepler and TESS datasets
produced by `src/data_processing.py`.

**Goals:**
- Load the processed CSVs from `data/processed/`.
- Optionally re-run the cleaning functions from `src.data_processing` on the raw files.
- Inspect shapes, columns, and label distributions.
- Ensure the data alignment (Kepler vs. TESS) looks correct before moving on.

> Note: This notebook is meant for lightweight validation only.  
> More detailed EDA and visualisations are in `03_eda_visualisation.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# add project root (one level up from notebooks/) to import src
PROJECT_ROOT = Path("..").resolve()
#print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import clean_kepler_df, clean_tess_df  
import pandas as pd

In [ ]:
RAW = PROJECT_ROOT / "data" / "raw"
#print(RAW)

read_opts = dict(engine="python", sep=None, comment="#", na_values=["", "NaN", "--", " "])

kepler_raw_path = next(RAW.glob("*cumulative*.csv"))
tess_raw_path   = next(RAW.glob("*TOI*.csv"))

df_kepler = pd.read_csv(kepler_raw_path, **read_opts)
df_tess   = pd.read_csv(tess_raw_path, **read_opts)

df_kepler.shape, df_tess.shape

In [ ]:
df_kepler_clean = clean_kepler_df(df_kepler)
df_tess_clean   = clean_tess_df(df_tess)  # drops KP by default and adds disposition_aligned

df_kepler_clean.head(), df_tess_clean.head()

In [ ]:
print("Kepler dispositions:")
print(df_kepler_clean["disposition"].value_counts())

print("\nTESS dispositions (aligned):")
print(df_tess_clean["disposition_aligned"].value_counts())

In [ ]:
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df_kepler_clean.to_csv(PROCESSED / "kepler_clean.csv", index=False)
df_tess_clean.to_csv(PROCESSED / "tess_clean.csv", index=False)

(PROCESSED / "kepler_clean.csv").stat().st_size, (PROCESSED / "tess_clean.csv").stat().st_size